# Alaska salmon data cleaning

This notebook holds the code to clean the Alaskan salmon dataset found on [KNB](https://knb.ecoinformatics.org/view/doi:10.5063/F1707ZTM). This data set spans 1922 to 2017 and contains salmon measurements, sample locations, and type of salmon capture. The final result is a local relational database that makes this data more usable.

In [1]:
# import in libraries
import pandas as pd
import numpy as np
import os
import janitor

In [2]:
# load in data
salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")
length_type = pd.read_csv("Alaskan-salmon1922-2017/data/length_type_lookup.csv")
location = pd.read_csv("Alaskan-salmon1922-2017/data/Locations_subdistricts_uniqueID.csv")
project_type = pd.read_csv("Alaskan-salmon1922-2017/data/ASLProjectType.csv")
gear =  pd.read_csv("Alaskan-salmon1922-2017/data/gear.csv")


/var/folders/kj/1ybgv25d7zd06v9ndqdccd2r0000gn/T/ipykernel_36242/1368001829.py:2: DtypeWarning: Columns (1,3,8,11,12,13,15,16,17,18,26) have mixed types. Specify dtype option on import or set low_memory=False.
  salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")


Look at the datatypes of the data.

In [3]:
salmon_data.dtypes

Species                      object
Length.Measurement.Type      object
sampleYear                  float64
ASLProjectType               object
LocationID                   object
sampleDate                   object
Length                      float64
Weight                      float64
Sex                          object
Salt.Water.Age              float64
DataSource                   object
cardNo                       object
fishNum                      object
Age.Error                    object
Fresh.Water.Age             float64
Sex.Determination.Method     object
subSystem                    object
Flag                         object
Gear                         object
SASAP.Region                 object
LocationUnique               object
DistrictID                  float64
Sub.DistrictID              float64
Stat.area                   float64
Lat                         float64
Lon                         float64
AWC_CODE                     object
dtype: object

### The first step is to make a primary key for our main dataframe
Add an ID column. 

In [4]:
# add an id column to the salmon dataframe
salmon_data['salmon_id'] = salmon_data.index

In [5]:
# initial look at the dataframe
salmon_data.head()

,Species,Length.Measurement.Type,sampleYear,ASLProjectType,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,...,Gear,SASAP.Region,LocationUnique,DistrictID,Sub.DistrictID,Stat.area,Lat,Lon,AWC_CODE,salmon_id
0,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,0
1,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,1
2,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,2
3,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,3
4,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,4.0,...,troll,Southeast,Affleck Canal/Spanish Is/Louise Cove-commercia...,105.0,10.0,10510.0,NaN,NaN,NaN,4


Check the new datatypes.

In [6]:
salmon_data.dtypes

Species                      object
Length.Measurement.Type      object
sampleYear                  float64
ASLProjectType               object
LocationID                   object
sampleDate                   object
Length                      float64
Weight                      float64
Sex                          object
Salt.Water.Age              float64
DataSource                   object
cardNo                       object
fishNum                      object
Age.Error                    object
Fresh.Water.Age             float64
Sex.Determination.Method     object
subSystem                    object
Flag                         object
Gear                         object
SASAP.Region                 object
LocationUnique               object
DistrictID                  float64
Sub.DistrictID              float64
Stat.area                   float64
Lat                         float64
Lon                         float64
AWC_CODE                     object
salmon_id                   

Yay! It looks like all of the data is in the right data type except for the sampleYear and sampleDate. 
 

In [7]:
# convert salmon year to an int
salmon_data['sampleYear'] = salmon_data['sampleYear'].astype('Int64')

In [8]:
# view the structure of the date column
salmon_data['sampleDate']

0           1992-03-31
1           1992-03-31
2           1992-03-31
3           1992-03-31
4           1992-03-31
               ...    
14347456    2015-05-27
14347457    2015-05-27
14347458    2015-05-27
14347459    2015-05-27
14347460    2015-05-27
Name: sampleDate, Length: 14347461, dtype: object

All of the dates seem like their in the same format. We'll attempt to give it a date type when we make the relational database but we'll revisit it if there's an issue. 

We don't need all of the columns for this analysis so we can subset. 

In [9]:
salmon_data = salmon_data[['Species', 'Length.Measurement.Type', 'sampleYear', 'ASLProjectType', 'LocationID', 'sampleDate', 'Length', 'Weight', 'Sex','Salt.Water.Age', 'Fresh.Water.Age', 'Gear', 'Stat.area', 'DataSource']]

In [10]:
salmon_data

,Species,Length.Measurement.Type,sampleYear,ASLProjectType,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,Fresh.Water.Age,Gear,Stat.area,DataSource
0,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward
1,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,0.0,troll,10510.0,ADFG Southeast and Westward
2,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward
3,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,0.0,troll,10510.0,ADFG Southeast and Westward
4,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,4.0,0.0,troll,10510.0,ADFG Southeast and Westward
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14347456,sockeye,NaN,2015,test fishing,Karluk River,2015-05-27,NaN,NaN,NaN,2.0,2.0,seine,25520.0,ADFG Southeast and Westward
14347457,sockeye,NaN,2015,test fishing,Karluk River,2015-05-27,NaN,NaN,NaN,2.0,0.0,seine,25520.0,ADFG Southeast and Westward
14347458,sockeye,NaN,2015,test fishing,Karluk River,2015-05-27,NaN,NaN,NaN,2.0,2.0,seine,25520.0,ADFG Southeast and Westward
14347459,sockeye,NaN,2015,test fishing,Karluk River,2015-05-27,NaN,NaN,NaN,2.0,2.0,seine,25520.0,ADFG Southeast and Westward


In [11]:
salmon_data['LocationID'].unique()

array(['Affleck Canal/Spanish Is/Louise Cove', 'Alava To Sykes',
       'Alsek River', ..., 'Blue Bill Lake', 'Three Hills', 'Upper Thumb'],
      dtype=object)

# Length dataset cleaning 

In [12]:
length_type.head()

,Length.Measurement.Type,Length_fill,LocationID,Species,ASLProjectType,sampleYear
0,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1998.0
1,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1999.0
2,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,2000.0
3,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2003.0
4,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2004.0


In [13]:
length_type.dtypes

Length.Measurement.Type     object
Length_fill                 object
LocationID                  object
Species                     object
ASLProjectType              object
sampleYear                 float64
dtype: object

In [14]:
# check all of the unique values of the year column
length_type['sampleYear'].unique()

array([1998., 1999., 2000., 2003., 2004., 1984., 1983., 1985., 1986.,
       1987., 1988., 1990., 1991., 1992., 1993., 2002., 2005., 2006.,
       2007., 2008., 1989., 1996., 2010., 2011., 2013., 2014., 1994.,
       1995., 1997., 2001., 2009., 2012., 2015., 1982., 1965., 1967.,
       1964., 1970., 1971., 1972., 1974., 1975., 1963., 1966., 1968.,
       1969., 1973.,   nan, 1976., 1960., 1978., 2016., 1979., 1980.,
       1981., 1977., 1961., 2017., 1962., 1959., 1958., 1957.])

In [15]:
# change year from float ot int
length_type['sampleYear'] = length_type['sampleYear'].astype('Int64')

In [16]:
location.dtypes

SASAP.Region               object
SASAP.Region_Corrected     object
Location                   object
ASLProjectType             object
District                   object
Sub.District               object
LocationUnique             object
DistrictID                float64
Sub.DistrictID            float64
Stat.area                 float64
Lat                       float64
Lon                       float64
AWC_CODE                   object
source                     object
LocationID                 object
dtype: object

In [17]:
location.head()

,SASAP.Region,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,LocationUnique,DistrictID,Sub.DistrictID,Stat.area,Lat,Lon,AWC_CODE,source,LocationID
0,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,Aleutians-commercial catch-302,302.0,NaN,NaN,NaN,NaN,NaN,NaN,Aleutians
1,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,Bear River-commercial catch-31511,315.0,11.0,31511.0,NaN,NaN,NaN,NaN,Bear River
2,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,Bear River-escapement-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
3,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
4,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River


In [18]:
# subset for only necessary rows
location = location[['LocationUnique', 'SASAP.Region_Corrected','Location', 'ASLProjectType', 'District', 'Sub.District', 'Lat', 'Lon', 'LocationID']]

In [19]:
location['Location'].unique()

array([nan, 'Bear River', 'Bear River (genetics)', ...,
       'Y6 (Subdistrict 6)',
       'Yukon Crossing, Y.T., Canada (Village/City)', 'Yukon District'],
      dtype=object)

In [20]:
location_rename = location.rename(columns = {'SASAP.Region_Corrected' : 'LocationID'})

In [21]:
location_rename

,LocationUnique,LocationID,Location,ASLProjectType,District,Sub.District,Lat,Lon,LocationID
0,Aleutians-commercial catch-302,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,NaN,NaN,Aleutians
1,Bear River-commercial catch-31511,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,NaN,NaN,Bear River
2,Bear River-escapement-31511,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,56.0389,-160.2734,Bear River
3,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,56.0389,-160.2734,Bear River
4,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,56.0389,-160.2734,Bear River
...,...,...,...,...,...,...,...,...,...
2782,Y6 (Subdistrict 6)-subsistence catch-33460,Yukon,Y6 (Subdistrict 6),subsistence catch,Yukon,Y6 (Subdistrict 6),NaN,NaN,Y6 (Subdistrict 6)
2783,"Yukon Crossing, Y.T., Canada (Village/City)-co...",Yukon,"Yukon Crossing, Y.T., Canada (Village/City)",commercial catch,Yukon,NaN,NaN,NaN,"Yukon Crossing, Y.T., Canada (Village/City)"
2784,"Yukon Crossing, Y.T., Canada (Village/City)-es...",Yukon,"Yukon Crossing, Y.T., Canada (Village/City)",escapement,Yukon,NaN,62.3540,-136.5010,"Yukon Crossing, Y.T., Canada (Village/City)"
2785,Yukon District-subsistence catch-NA,Yukon,Yukon District,subsistence catch,Yukon,NaN,NaN,NaN,Yukon District


In [22]:
gear.head()

,Gear,SASAP.Gear
0,NaN,NaN
1,handpicked or carcass,handpicked or carcass
2,beach seine,seine
3,sport hook and line,sport hook and line
4,weir,weir


In [23]:
location.head()

,LocationUnique,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,Lat,Lon,LocationID
0,Aleutians-commercial catch-302,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,NaN,NaN,Aleutians
1,Bear River-commercial catch-31511,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,NaN,NaN,Bear River
2,Bear River-escapement-31511,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,56.0389,-160.2734,Bear River
3,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,56.0389,-160.2734,Bear River
4,Bear River-test fishing-31511,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,56.0389,-160.2734,Bear River


In [24]:
# join the salmon data with the location data to get the unique location column
salmon_data = pd.merge(salmon_data, location, how = "inner", on = 'LocationID')

In [25]:
salmon_data.head()

,Species,Length.Measurement.Type,sampleYear,ASLProjectType_x,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,...,Stat.area,DataSource,LocationUnique,SASAP.Region_Corrected,Location,ASLProjectType_y,District,Sub.District,Lat,Lon
0,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Test Seine,test fishing,105,10,NaN,NaN
1,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Test Troll,test fishing,105,10,NaN,NaN
2,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...,Southeast,District 105 Traditional Seine,commercial catch,105,10,NaN,NaN
3,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...,Southeast,District 105 Traditional Troll,commercial catch,105,10,NaN,NaN
4,chinook,length not taken,1992,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,...,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...,Southeast,District 105 Troll Sublegal,test fishing,105,10,NaN,NaN


In [26]:
# reselect the columns we need
salmon_data = salmon_data[['Species', 'Length.Measurement.Type', 'sampleYear', 'ASLProjectType_y', 'sampleDate', 'Length', 'Weight', 'Sex','Salt.Water.Age', 'Fresh.Water.Age', 'Gear', 'Stat.area', 'DataSource', 'LocationUnique']]

# clean column names
salmon_data = salmon_data.clean_names()

# rename the columns
salmon_data = salmon_data.rename(columns = {'aslprojecttype_y' : 'asl_project_type'})

In [27]:
salmon_data.head()

,species,length_measurement_type,sampleyear,asl_project_type,sampledate,length,weight,sex,salt_water_age,fresh_water_age,gear,stat_area,datasource,locationunique
0,chinook,length not taken,1992,test fishing,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...
1,chinook,length not taken,1992,test fishing,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...
2,chinook,length not taken,1992,commercial catch,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...
3,chinook,length not taken,1992,commercial catch,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-commercia...
4,chinook,length not taken,1992,test fishing,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward,Affleck Canal/Spanish Is/Louise Cove-test fish...


In [29]:
salmon_data.dtypes

species                     object
length_measurement_type     object
sampleyear                   Int64
asl_project_type            object
sampledate                  object
length                     float64
weight                     float64
sex                         object
salt_water_age             float64
fresh_water_age            float64
gear                        object
stat_area                  float64
datasource                  object
locationunique              object
dtype: object

In [34]:
# look at the unique values
print(salmon_data['species'].unique())
print(salmon_data['length_measurement_type'].unique())
print(salmon_data['sampleyear'].unique())
print(salmon_data['asl_project_type'].unique())
print(salmon_data['sampledate'].unique())
print(salmon_data['length'].unique())
print(salmon_data['weight'].unique())
print(salmon_data['sex'].unique())
print(salmon_data['salt_water_age'].unique())
print(salmon_data['fresh_water_age'].unique())
print(salmon_data['gear'].unique())
print(salmon_data['stat_area'].unique())
print(salmon_data['locationunique'].unique())

['chinook' 'chum' 'coho' 'pink' 'sockeye']
['length not taken' 'mid-eye to fork of tail' 'mid-eye to hypural plate'
 'post orbit to hypural plate' 'post-orbit to fork of tail'
 'tip of snout to fork of tail' 'tip of snout to tip of tail' 'unknown'
 nan 'cleithral arch to fork']
<IntegerArray>
[1992, 2002, 1997, 1999, 2000, 2001, 2003, 1996, 1991, 1995, 2007, 1998, 1969,
 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, <NA>, 1979, 1980, 1981,
 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1993, 1994, 2004, 2005,
 2006, 2008, 2009, 2010, 2011, 2012, 1966, 1968, 2013, 2014, 2015, 2016, 1964,
 1965, 1967, 2017, 1961, 1959, 1962, 1963, 1957, 1960, 2018, 1937, 1958]
Length: 64, dtype: Int64
['test fishing' 'commercial catch' 'escapement' 'subsistence catch' nan
 'brood excess' 'brood stock' 'hatchery cost recovery' 'sport catch'
 'personal use' 'test fish' 'unknown' 'commercial common property'
 'mark/recapture' 'sport fish']
['1992-03-31' '2002-07-15' '2002-08-30' ... '1977-05

In [35]:
# change weird unknowns to NAs
salmon_data.loc[salmon_data['length_measurement_type']== 'unknown', 'length_measurement_type'] = np.nan
print(salmon_data['length_measurement_type'].unique())

['length not taken' 'mid-eye to fork of tail' 'mid-eye to hypural plate'
 'post orbit to hypural plate' 'post-orbit to fork of tail'
 'tip of snout to fork of tail' 'tip of snout to tip of tail' nan
 'cleithral arch to fork']


In [36]:
# change weird unknowns to NA
salmon_data.loc[salmon_data['asl_project_type']== 'unknown', 'asl_project_type'] = np.nan
print(salmon_data['asl_project_type'].unique())

['test fishing' 'commercial catch' 'escapement' 'subsistence catch' nan
 'brood excess' 'brood stock' 'hatchery cost recovery' 'sport catch'
 'personal use' 'test fish' 'commercial common property' 'mark/recapture'
 'sport fish']


In [ ]:
# change weird unknowns to NA
# filter rows that are not 'male' and not 'female'
invalid_mask = (~salmon_data['sex'].isin(['male', 'female']))

# fill these rows with NA
salmon_data.loc[invalid_mask, 'sex'] = np.nan

print(salmon_data['sex'].unique())

[nan 'male' 'female']


In [41]:
# save data to a csv file
salmon_data.to_csv('salmon_data_clean.csv')